# Tuberculosis Detection with TBX11K Dataset

This notebook demonstrates how to use the object detection model trained on the TBX11K dataset for detecting tuberculosis in chest X-rays.

In [ ]:
import os
import sys
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

# Ensure the tuberculosis module is in the path
sys.path.append(os.path.abspath('../'))

from tuberculosis.tb_dataset import TBX11KDataset
from tuberculosis.tb_detect_bbox import load_tb_detection_model, detect_tb, visualize_tb_detection, create_heatmap_overlay

## Load the TB Detection Model

In [ ]:
# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Path to the trained model
model_path = "../tuberculosis/models/detection/tb_detector_final.pth"

# Number of TB-related classes (typically 3: TB, Suspicious, Other)
num_classes = 3

try:
    # Load the model
    model = load_tb_detection_model(model_path, num_classes, device)
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    print("\nNote: You need to train the model first using tb_train_detector.py")

## Create a Test Dataset or Load a Sample Image

We can either test on a single image or create a dataset for batch testing.

In [ ]:
# Option 1: Load a single image
test_image_path = "../tb_data/TestImages/sample.png"  # Replace with your image path

# Check if the image exists
if os.path.exists(test_image_path):
    print(f"Test image found: {test_image_path}")
    
    # Display the image
    img = Image.open(test_image_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.title("Test Image")
    plt.axis('off')
    plt.show()
else:
    print(f"Test image not found at {test_image_path}")

In [ ]:
# Option 2: Create a dataset from TBX11K annotations
try:
    from torchvision.transforms import ToTensor, Normalize, Compose
    
    # Paths to dataset
    ann_file = "../tb_data/annotations/tbx11k_test.json"  # Replace with your annotation file
    img_prefix = "../tb_data/images/"  # Replace with your image directory
    
    # Check if the annotation file exists
    if os.path.exists(ann_file):
        transforms = Compose([
            ToTensor(),
            Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        test_dataset = TBX11KDataset(ann_file, img_prefix, transforms=transforms)
        print(f"Dataset created with {len(test_dataset)} images")
        
        # Get the category names
        category_names = test_dataset.get_category_names()
        print(f"Categories: {category_names}")
    else:
        print(f"Annotation file not found at {ann_file}")
except Exception as e:
    print(f"Error creating dataset: {e}")

## Run Detection on the Test Image

In [ ]:
# Run detection on the test image
if 'model' in locals() and os.path.exists(test_image_path):
    # Detect TB in the image
    detections = detect_tb(model, test_image_path, score_threshold=0.5)
    
    # Visualize the detections
    visualize_tb_detection(detections)
    
    # Create a heatmap overlay
    heatmap = create_heatmap_overlay(detections)
    
    # Display the heatmap
    plt.figure(figsize=(10, 8))
    plt.imshow(heatmap)
    plt.title("TB Detection Heatmap")
    plt.axis('off')
    plt.show()
else:
    print("Cannot run detection. Make sure the model is loaded and the test image exists.")

## Process Multiple Images from the Dataset

In [ ]:
# Process multiple images if dataset is available
if 'test_dataset' in locals() and 'model' in locals():
    from torch.utils.data import DataLoader
    import random
    
    # Number of images to process
    num_images = min(5, len(test_dataset))
    
    # Choose random indices
    indices = random.sample(range(len(test_dataset)), num_images)
    
    for idx in indices:
        sample = test_dataset[idx]
        image = sample['image']
        target = sample['target']
        
        # Get image path
        img_id = target['image_id'].item()
        img_info = test_dataset.coco.loadImgs(img_id)[0]
        img_path = os.path.join(test_dataset.img_prefix, img_info['file_name'])
        
        print(f"Processing image: {img_path}")
        
        # Run detection
        detections = detect_tb(model, img_path, score_threshold=0.3)
        
        # Visualize detections
        visualize_tb_detection(detections, category_names=category_names)
else:
    print("Dataset or model not available for batch processing.")

## Combine Object Detection with Segmentation for Enhanced TB Analysis

In [ ]:
# Load both detection and segmentation models for comprehensive analysis
if 'model' in locals() and os.path.exists(test_image_path):
    try:
        # Import the segmentation model (from our previous TB segmentation code)
        from tuberculosis.tb_detect import process_image
        
        print("Running combined detection and segmentation analysis...")
        
        # Run object detection
        detections = detect_tb(model, test_image_path, score_threshold=0.4)
        
        # Run segmentation
        segmentation_results = process_image(test_image_path)
        
        # Display combined results
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        # Original image
        axes[0].imshow(detections['image'])
        axes[0].set_title("Original Chest X-ray")
        axes[0].axis('off')
        
        # Detection results
        axes[1].imshow(detections['image'])
        axes[1].set_title("TB Detection (Object Detection)")
        axes[1].axis('off')
        
        # Draw boxes
        for box, label, score in zip(detections['boxes'].numpy(), 
                                    detections['labels'].numpy(), 
                                    detections['scores'].numpy()):
            x1, y1, x2, y2 = box.astype(int)
            rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, 
                                 edgecolor='red', linewidth=2)
            axes[1].add_patch(rect)
            axes[1].text(x1, y1-5, f"TB: {score:.2f}", 
                        bbox=dict(facecolor='red', alpha=0.5),
                        fontsize=8, color='white')
        
        # Segmentation results - use the probability map from segmentation
        im = axes[2].imshow(segmentation_results['probability_map'], cmap='hot')
        axes[2].set_title(f"TB Segmentation (Lesion Area: {segmentation_results['lesion_percentage']:.2f}%)")
        axes[2].axis('off')
        fig.colorbar(im, ax=axes[2], shrink=0.6)
        
        plt.tight_layout()
        plt.show()
        
        # Print integrated results
        print("\nIntegrated Analysis Results:")
        print(f"Number of TB regions detected: {len(detections['boxes'])}")
        print(f"TB lesion area percentage: {segmentation_results['lesion_percentage']:.2f}%")
        print(f"TB severity classification: {segmentation_results['severity']}")
        
        # Check if detection and segmentation agree
        detection_positive = len(detections['boxes']) > 0
        segmentation_positive = segmentation_results['detected']
        
        if detection_positive and segmentation_positive:
            print("\nConclusion: TB DETECTED ⚠️ (confirmed by both detection and segmentation)")
        elif detection_positive and not segmentation_positive:
            print("\nConclusion: SUSPICIOUS ⚠️ (detected by object detection only)")
        elif not detection_positive and segmentation_positive:
            print("\nConclusion: SUSPICIOUS ⚠️ (detected by segmentation only)")
        else:
            print("\nConclusion: NO TB DETECTED ✓")
        
    except Exception as e:
        print(f"Error in combined analysis: {e}")
else:
    print("Cannot run combined analysis. Make sure both models are available.")